# Building phylogenies from scratch

Fill in the short `TODO` gaps before running the later cells.

We want to learn how to do linguistic phylogenies, as they are some of the most straightforward examples of applying tree-like methods to culture. 

The three CSVs are unmodified tables from [IE-CoR v1.1](https://github.com/lexibank/iecor/tree/dd1ebc997bc38c824fb18a1ef234bbf2c229420b/cldf) (Heggarty, Anderson & Scarborough 2024). We start by loading and installing the necessary packages. 

In [ ]:
%pip -q install lingpy==2.6.14 

In [ ]:
from itertools import combinations
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import (pdist, 
                                    squareform)
from scipy.cluster.hierarchy import (linkage, 
                                     dendrogram, 
                                     cophenet)
from scipy.stats import spearmanr
from lingpy import LexStat

sns.set(context="paper", 
        style="ticks", 
        font_scale=1, 
        palette="colorblind")

## 1. Inspect the raw Lexibank tables

Each row in `iecor-forms.csv` is a word for one language and one meaning. `Phonemic` records an IPA-style pronunciation; `Phonemic_Segments` separates its sounds for LingPy. The source tables also contain language locations and meaning names. Run this notebook from `S5`, where the three CSVs are stored. In Colab, upload the CSVs first.

In [ ]:
forms = pd.read_csv(
    "iecor-forms.csv",
    dtype={"Language_ID": str, "Parameter_ID": str}
)

languages = pd.read_csv(
    "iecor-languages.csv",
    dtype={"ID": str}
).set_index("ID")

meanings = pd.read_csv(
    "iecor-parameters.csv",
    dtype={"ID": str}
).set_index("ID")

forms.loc[forms.Phonemic.notna(),
          ["Language_ID", "Parameter_ID", "Form", 
           "Phonemic", "Phonemic_Segments"]].head()

Let's look at how head is expressed in a couple of languages:

In [ ]:
# IE-CoR IDs 25, 39, and 68 are French, Italian, and Spanish.
head_id = meanings.index[meanings.Name == "head"][0]
forms.loc[(forms.Parameter_ID == head_id) & forms.Language_ID.isin(["25", "39", "68"]),
          ["Language_ID", "Form", "Phonemic", "Phonemic_Segments"]]

## 2. Make a comparable wordlist

Keep present-day Romance varieties with phonemic transcriptions. Two present-day varieties in the source have no `Phonemic_Segments`, so they cannot enter this sound-based exercise. Some meanings have missing forms or synonyms. To make every language pair comparable over the same meanings, keep only meanings with **exactly one usable form in every selected variety**. This is a transparent teaching choice; a larger analysis could handle missing forms and synonyms.

In [ ]:
# Start with present-day Romance varieties.
romance = languages[(languages.Clade == "Italic;Roman") & ~languages.historical]

# LingPy needs both the pronunciation and its separated sound segments.
usable = forms[forms.Language_ID.isin(romance.index) &
               forms.Phonemic.notna() & forms.Phonemic_Segments.notna()]
romance = romance[romance.index.isin(usable.Language_ID)]

# Keep meanings with exactly one form in every remaining variety.
counts = (usable.groupby(["Parameter_ID", "Language_ID"]).size()
          .unstack(fill_value=0).reindex(columns=romance.index, fill_value=0))
shared_meanings = counts.index[counts.eq(1).all(axis=1)]
selected = usable[usable.Parameter_ID.isin(shared_meanings)]
romance[["Name", "Latitude", "Longitude"]]

## 3. Let LingPy propose cognate sets

We can use LingPy to propose cognates -- words in different languages that descend from the same earlier word. That way we can measure similarity between languages later. Start by getting the data into LingPy:

In [ ]:
# LingPy's four columns: language, meaning, IPA form, and sound tokens.
entries = {0: ["doculect", "concept", "ipa", "tokens"]}
for i, row in enumerate(selected.itertuples(index=False), 1):
    entries[i] = [romance.loc[row.Language_ID, "Name"],
                  meanings.loc[row.Parameter_ID, "Name"],
                  row.Phonemic, row.Phonemic_Segments.split()]

logging.disable(logging.CRITICAL)  # Hide LingPy's sound-model setup messages.
lex = LexStat(entries, no_bscorer=True)
print(f"LingPy loaded {len(lex)} forms in {lex.height} meanings and {lex.width} varieties.")

Now you can apply their algorithm to get cognates:

In [ ]:
# SCA groups similar-sounding forms within each meaning.
lex.cluster(method=..., threshold=0.6, ref="auto")  # TODO: choose SCA
logging.disable(logging.NOTSET)
print(f"Proposed {len({lex[i, 'auto'] for i in lex})} cognate sets.")

Each number labels a proposed cognate set, not a verified historical judgment. Let's look at the meaning "heart".

In [ ]:
pd.DataFrame([{"Language": lex[i, "doculect"],
               "Word": lex[i, "ipa"],
               "Automatic cognate set": lex[i, "auto"]}
              for i in lex.get_list(row="heart", flat=True)])

## 4. From cognates to a tree

For each language, we have 150 words for 150 meanings. We can use cognate lists to measure similarity between languages by counting overlaps in cognates.

In [ ]:
# Each row is a meaning; each column is a variety; numbers label cognate sets.
cognates = pd.DataFrame(
    [(lex[i, "concept"], lex[i, "doculect"], lex[i, "auto"]) for i in lex],
    columns=["Meaning", "Language", "Cognate set"]
).pivot(index="Meaning", columns="Language", values="Cognate set")
cognates = cognates[romance.Name]  # Preserve the source variety order.
cognates.head()

Let's compute the pairwise distances between varieties based on their cognate sets. We'll use the Hamming distance, which gives the fraction of meanings with different cognate sets.

In [ ]:
# Hamming distance is the fraction of meanings with different cognate IDs.
############################################################
lexical_distances = ...  # TODO: Hamming distances between variety columns (use pdist)
############################################################
lexical_matrix = pd.DataFrame(squareform(lexical_distances),
                              index=cognates.columns, columns=cognates.columns)
lexical_matrix.round(2)

Now we plot the distance matrix:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(lexical_matrix, cmap="cividis", square=True,
            cbar_kws={"label": "Fraction of meanings with different cognate sets"}, ax=ax)
ax.tick_params(axis="x", labelrotation=90)
ax.tick_params(axis="y", labelrotation=0)
plt.tight_layout()
plt.show()

Finally, we can use a tree algorithm to transform this distance matrix to a tree. **UPGMA** (average linkage) joins the nearest groups and averages distances between groups. The tree's horizontal scale is lexical distance, **not time**.

In [ ]:
# Average linkage turns the pairwise distances into a UPGMA tree.
############################################################
tree = ...  # TODO: average linkage from lexical_distances (use linkage)
############################################################
fig, ax = plt.subplots(figsize=(8, 6))
dendrogram(tree, labels=cognates.columns, orientation="right",
           color_threshold=0, above_threshold_color="black", ax=ax)
ax.set_xlabel("Fraction of different cognate sets")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

## 5. Compare tree distance with geographic distance

We might expect that the geographical distance between languages in our dataset matches the phylogenetic distance. We need a function to measure distances on the map:

In [ ]:
# Haversine distance follows the Earth's surface (Earth radius ≈ 6,371 km).
def great_circle_km(a, b):
    lat1, lon1 = np.radians(a)
    lat2, lon2 = np.radians(b)
    h = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * 6371 * np.arcsin(np.sqrt(h))

Now let's compute the geographic distance:

In [ ]:
# Match the variety order used for the cognate matrix and tree.
locations = romance.set_index("Name").loc[cognates.columns, ["Latitude", "Longitude"]]
############################################################
geographic_distances = ...  # TODO: great-circle distances between locations (use pdist again)
############################################################
tree_distances = 2 * cophenet(tree)  # Tip-to-tip path is twice the joining height.

# combinations() and pdist() list pairs in the same order.
pairs = pd.DataFrame(combinations(cognates.columns, 2),
                     columns=["Variety A", "Variety B"])
pairs["Geographic km"] = geographic_distances
pairs["Tree distance"] = tree_distances
pairs.sort_values("Geographic km").head(10).round(2)

Finally, we can compute the Spearman correlation. What does it tell you?

In [ ]:
rho = ... # TODO: compute Spearman's rho between geographic_distances and tree_distances

And plot the relationship between distances:

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(geographic_distances, tree_distances, s=26, alpha=0.65)
ax.text(0.96, 0.05, f"Spearman ρ = {rho:.2f}",
        transform=ax.transAxes, ha="right", va="bottom")
ax.set(xlabel="Geographic distance (km)",
       ylabel="Tree path length (lexical-distance units)")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()